# Six-Frame Translation and ORF Finder

**Author:** Muhammad Junaid

## Project aim

This notebook is a small dry-lab bioinformatics project for learning how coding regions can be explored from a DNA sequence.

**Workflow:** DNA sequence → validation → reverse complement → six reading frames → translation → candidate ORFs → coordinates → Biopython check.

The six reading frames are +1, +2, +3 on the forward strand and -1, -2, -3 on the reverse-complement strand.

An ORF found by this simple method is a **candidate ORF**. Its presence alone does not prove that it is a functional gene.


In [ ]:
CODON_TABLE = {
    'ATA':'I','ATC':'I','ATT':'I','ATG':'M','ACA':'T','ACC':'T','ACG':'T','ACT':'T',
    'AAC':'N','AAT':'N','AAA':'K','AAG':'K','AGC':'S','AGT':'S','AGA':'R','AGG':'R',
    'CTA':'L','CTC':'L','CTG':'L','CTT':'L','CCA':'P','CCC':'P','CCG':'P','CCT':'P',
    'CAC':'H','CAT':'H','CAA':'Q','CAG':'Q','CGA':'R','CGC':'R','CGG':'R','CGT':'R',
    'GTA':'V','GTC':'V','GTG':'V','GTT':'V','GCA':'A','GCC':'A','GCG':'A','GCT':'A',
    'GAC':'D','GAT':'D','GAA':'E','GAG':'E','GGA':'G','GGC':'G','GGG':'G','GGT':'G',
    'TCA':'S','TCC':'S','TCG':'S','TCT':'S','TTC':'F','TTT':'F','TTA':'L','TTG':'L',
    'TAC':'Y','TAT':'Y','TAA':'*','TAG':'*','TGC':'C','TGT':'C','TGA':'*','TGG':'W'
}

def validate_dna_sequence(dna_seq):
    """Check that a sequence is non-empty and contains valid DNA symbols."""
    dna_seq = dna_seq.upper().replace(" ", "").replace("\n", "")
    valid_bases = set("ACGTN")
    if not dna_seq:
        raise ValueError("The DNA sequence is empty.")
    invalid_bases = set(dna_seq) - valid_bases
    if invalid_bases:
        raise ValueError(f"Invalid DNA symbols found: {sorted(invalid_bases)}")
    return dna_seq

def get_reverse_complement(dna_seq):
    """Return the reverse-complement sequence."""
    complement = {'A':'T','T':'A','G':'C','C':'G','N':'N'}
    return "".join(complement[base] for base in reversed(dna_seq))

def translate_frame(dna_subseq):
    """Translate complete codons from the beginning of a sequence."""
    peptide = []
    for i in range(0, len(dna_subseq) - 2, 3):
        codon = dna_subseq[i:i+3]
        peptide.append(CODON_TABLE.get(codon, 'X'))
    return "".join(peptide)


## 1. ORF detection

The ORF finder looks for an `ATG` and then continues in the same reading frame until the next stop codon.

A minimum peptide length is used only as a simple filtering rule. It is **not** evidence that an ORF is biologically functional.


In [ ]:
def find_orfs_in_frame(dna_seq, strand, frame_number, min_len=30):
    """Find complete ATG-to-stop candidate ORFs in one reading frame."""
    results = []
    offset = frame_number - 1

    for codon_start in range(offset, len(dna_seq) - 2, 3):
        if dna_seq[codon_start:codon_start + 3] != "ATG":
            continue

        peptide = []
        stop_found = False

        for pos in range(codon_start, len(dna_seq) - 2, 3):
            codon = dna_seq[pos:pos + 3]
            amino_acid = CODON_TABLE.get(codon, "X")

            if amino_acid == "*":
                stop_found = True
                nucleotide_end = pos + 3
                stop_codon = codon
                break

            peptide.append(amino_acid)

        if stop_found and len(peptide) >= min_len:
            results.append({
                "strand": strand,
                "frame": f"{'+' if strand == 'forward' else '-'}{frame_number}",
                "start_nt": codon_start + 1,
                "end_nt": nucleotide_end,
                "length_nt": nucleotide_end - codon_start,
                "length_aa": len(peptide),
                "start_codon": "ATG",
                "stop_codon": stop_codon,
                "peptide": "".join(peptide)
            })

    return results


def scan_six_frames(dna_sequence, min_len=30):
    """Scan all three forward and three reverse-complement frames."""
    dna_sequence = validate_dna_sequence(dna_sequence)
    reverse_sequence = get_reverse_complement(dna_sequence)
    results = []

    for frame in range(1, 4):
        results.extend(find_orfs_in_frame(dna_sequence, "forward", frame, min_len))

    for frame in range(1, 4):
        results.extend(find_orfs_in_frame(reverse_sequence, "reverse", frame, min_len))

    return results


## 2. Test sequence: human HBB

For a controlled test, this notebook uses the human **HBB (hemoglobin beta)** coding sequence.

The expected coding frame is +1. The purpose here is to test and understand the pipeline, not to make a novel gene prediction.


In [ ]:
hbb_cds = (
    "ATGGTGCACCTGACTCCTGAGGAGAAGTCTGCCGTTACTGCCCTGTGGGGCAAGGTGAACGTGGATGAAG"
    "TTGGTGGTGAGGCCCTGGGCAGGCTGCTGGTGGTCTACCCTTGGACCCAGAGGTTCTTTGAGTCCTTTGG"
    "GGATCTGTCCACTCCTGATGCTGTTATGGGCAACCCTAAGGTGAAGGCTCATGGCAAGAAAGTGCTCGGT"
    "GCCTTTAGTGATGGCCTGGCTCACCTGGACAACCTCAAGGGCACCTTTGCCACACTGAGTGAGCTGCACT"
    "GTGACAAGCTGCACGTGGATCCTGAGAACTTCAGGCTCCTGGGCAACGTGCTGGTCTGTGTGCTGGCCCA"
    "TCACTTTGGCAAAGAATTCACCCCACCAGTGCAGGCTGCCTATCAGAAAGTGGTGGCTGGTGTGGCTAAT"
    "GCCCTGGCCCACAAGTATCACTAA"
)

hbb_cds = validate_dna_sequence(hbb_cds)

print("Sequence length:", len(hbb_cds), "nt")
print("First 30 nt:", hbb_cds[:30])
print("Last 30 nt :", hbb_cds[-30:])


## 3. Scan all six reading frames

The minimum candidate length is set to 30 amino acids for this learning exercise.

The output records the strand, frame, nucleotide coordinates, nucleotide length, peptide length, start codon, and stop codon.


In [ ]:
orf_results = scan_six_frames(hbb_cds, min_len=30)

print(f"{'Strand':<10} {'Frame':<7} {'Start':>7} {'End':>7} {'nt':>6} {'aa':>6} {'Start':<8} {'Stop':<6}")
print("-" * 70)

for orf in orf_results:
    print(
        f"{orf['strand']:<10} {orf['frame']:<7} "
        f"{orf['start_nt']:>7} {orf['end_nt']:>7} "
        f"{orf['length_nt']:>6} {orf['length_aa']:>6} "
        f"{orf['start_codon']:<8} {orf['stop_codon']:<6}"
    )

print("\nCandidate ORFs:", len(orf_results))


## 4. Inspect the candidates

A longer ORF is not automatically a functional gene. Length is only one sequence-level feature.

The table above is therefore treated as a list of **candidate ORFs**, not confirmed genes.


In [ ]:
for orf in sorted(orf_results, key=lambda x: x["length_aa"], reverse=True):
    print(
        f"{orf['frame']}: {orf['length_aa']} aa | "
        f"{orf['start_nt']}-{orf['end_nt']} nt | "
        f"{orf['start_codon']} -> {orf['stop_codon']}"
    )


## 5. Validate the expected HBB translation with Biopython

For the known HBB sequence, the expected coding frame is +1.

The custom peptide is compared directly with Biopython. This is a focused validation of the translation result for the expected frame.

It does **not** prove that every part of the ORF detection pipeline is biologically correct.


In [ ]:
from Bio.Seq import Seq

expected_hbb = next(orf for orf in orf_results if orf["frame"] == "+1")

custom_peptide = expected_hbb["peptide"]
biopython_peptide = str(Seq(hbb_cds).translate(to_stop=True))

print("Custom translation   :", custom_peptide)
print("Biopython translation:", biopython_peptide)
print("-" * 70)
print("Exact match:", custom_peptide == biopython_peptide)


## 6. Interpretation and limitations

The expected +1 frame produces the HBB coding peptide and matches the Biopython translation.

The six-frame scan may also produce other candidate ORFs above the chosen length threshold. These should **not** automatically be called functional genes.

This simple implementation does not perform full gene prediction or functional annotation.

### Limitations

- It uses a simple ATG-to-stop definition.
- It does not score ORFs using biological evidence.
- It does not compare candidates with protein databases.
- It does not handle alternative start codons.
- Ambiguous bases are accepted but may translate to `X`.
- Reverse-strand coordinates are relative to the reverse-complement sequence rather than converted back to original genomic coordinates.

These limitations are intentional for a learning-focused dry-lab project.


## 7. Skills demonstrated

- Python functions and dictionaries
- DNA sequence validation
- Reverse-complement generation
- Six-frame translation
- Basic ORF detection
- Nucleotide and peptide coordinates
- Biopython
- Validation of a custom implementation
- Biological interpretation and limitations

### Possible next step

A natural extension is to apply this workflow to an unknown FASTA sequence and add sequence-annotation or similarity-search evidence rather than treating an ORF as a confirmed gene.
